In [ ]:
import pandas as pd


DATA = REPO_PATH / "data"


tx = pd.read_csv(DATA / "walmart_transactions.csv")
stores = pd.read_csv(DATA / "store_locations.csv")


def normalize_store_id(values):
    codes = (
        values.astype("string")
        .str.upper()
        .str.extract(r"(\d{3})", expand=False)
    )
    return "FL-" + codes


def clean_transactions(raw):
    df = raw.copy()
    changes = []

    
    duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()

    changes.append({
        "decision": "Removed exact duplicate rows",
        "rows_affected": duplicates
    })

  
    prices = df["unit_price"].astype("string")

    prices_with_dollar = int(
        prices.str.contains("$", regex=False, na=False).sum()
    )

    df["unit_price"] = pd.to_numeric(
        prices.str.replace("$", "", regex=False)
              .str.replace(",", "", regex=False)
              .str.strip(),
        errors="coerce"
    )

    changes.append({
        "decision": "Removed $ and converted prices to numeric",
        "rows_affected": prices_with_dollar
    })

   
    df["quantity"] = pd.to_numeric(
        df["quantity"], errors="coerce"
    )

    negative_qty = int((df["quantity"] < 0).sum())
    df["is_return"] = df["quantity"] < 0

    changes.append({
        "decision": "Kept negative quantities as potential returns",
        "rows_affected": negative_qty
    })

 
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="mixed", errors="coerce"
    )

    invalid_dates = int(df["timestamp"].isna().sum())
    df = df.dropna(subset=["timestamp"]).copy()

    df["date"] = df["timestamp"].dt.normalize()

    changes.append({
        "decision": "Dropped rows with invalid timestamps",
        "rows_affected": invalid_dates
    })

    old_ids = df["store_id"].astype("string")
    df["store_id"] = normalize_store_id(df["store_id"])

    fixed_ids = int(
        (old_ids.fillna("") !=
         df["store_id"].fillna("")).sum()
    )

    changes.append({
        "decision": "Standardized store IDs to FL-###",
        "rows_affected": fixed_ids
    })

    return df, pd.DataFrame(changes)



clean_tx, decision_log = clean_transactions(tx)


stores["store_id"] = normalize_store_id(stores["store_id"])

print("Rows before cleaning:", len(tx))
print("Rows after cleaning:", len(clean_tx))

print(
    "Transaction rows without a matching store:",
    (~clean_tx["store_id"].isin(stores["store_id"])).sum()
)

display(clean_tx.head())



In [ ]:
print("Cleaning Decision Log")
display(decision_log)

print("Total decisions recorded:", len(decision_log))

In [ ]:
catalog = pd.read_csv(DATA / "product_catalog.csv")

poptart_map = {
    "PT-12": "Pop-Tarts Strawberry",
    "SKU#459812": "Pop-Tarts Strawberry",
    "POPTART-STRAW": "Pop-Tarts Strawberry",
    "PT_STRAWBERRY": "Pop-Tarts Strawberry"
}


catalog["canonical_product"] = (
    catalog["sku"].replace(poptart_map)
)


original_skus = catalog["sku"].nunique()
partial_groups = catalog["canonical_product"].nunique()

print("Original distinct SKUs:", original_skus)
print("Groups after partial consolidation:", partial_groups)

group_sizes = (
    catalog.groupby("canonical_product")["sku"]
    .nunique()
    .sort_values(ascending=False)
)

print("Largest group:", group_sizes.index[0])
print("SKUs in group:", group_sizes.iloc[0])


display(
    catalog[catalog["sku"].isin(poptart_map)]
    [["sku", "product_name", "package_size",
      "canonical_product"]]
)

In [ ]:
import requests
import time

BASE = "https://archive-api.open-meteo.com/v1/archive"
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "daily": "precipitation_sum,wind_speed_10m_max",
        "timezone": "America/New_York"
    }

    for attempt in range(1, tries + 1):
        try:
            response = requests.get(
                BASE, params=params, timeout=10
            )

            response.raise_for_status()

            weather_df = pd.DataFrame(
                response.json()["daily"]
            )

            weather_df = weather_df.rename(
                columns={"time": "date"}
            )

            weather_df["date"] = pd.to_datetime(
                weather_df["date"]
            )

            weather_df["source"] = "open-meteo"

            return weather_df

        except requests.RequestException as error:
            if attempt == tries or (
                isinstance(error, requests.HTTPError)
                and response.status_code not in RETRY_STATUSES
            ):
                raise

            time.sleep(2 ** attempt)



weather = pd.DataFrame()

try:
    weather = daily_weather(
        "2024-09-01", "2024-09-30"
    )

    print("Weather rows:", len(weather))
    print("Timezone: America/New_York")
    print("Source:", weather["source"].iloc[0])

    display(weather.head())

except requests.RequestException as error:
    print("Weather API failed:", error)

In [ ]:
import matplotlib.pyplot as plt


daily_orders = (
    clean_tx.groupby("date")
    .size()
    .reset_index(name="orders")
    .sort_values("date")
)


plt.figure(figsize=(10, 5))

plt.plot(
    daily_orders["date"],
    daily_orders["orders"],
    marker="o",
    markersize=3
)


plt.axvline(
    pd.Timestamp("2024-09-26"),
    linestyle="--",
    label="Hurricane Helene landfall"
)

plt.title("Daily Walmart Orders Around Hurricane Helene")
plt.xlabel("Date")
plt.ylabel("Number of Orders")

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Days plotted:", len(daily_orders))
print("Total transaction rows:", daily_orders["orders"].sum())